# Solution F2 · Ridge with and without scaling: the same penalty is not the same model

**Type**: Compare

## Objective

Show that OLS is invariant to the scale of the features and Ridge is not, and measure **by how much** the same penalty
alpha shrinks features in different units, finding which alpha is needed to halve the effect of each one.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)
n = 200
Z = rng.normal(size=(n, 3))                            # three true variables, standard deviation 1
X = pd.DataFrame(Z * np.array([1.0, 30.0, 1000.0]), columns=["x1", "x2", "x3"])
y = 2 * Z.sum(axis=1) + rng.normal(size=n)             # same effect (2 per standard deviation) for all three
stds = X.std()
stds.round(1)

## Commented solution

In [ ]:
lin_raw = LinearRegression().fit(X, y)
lin_scaled = make_pipeline(StandardScaler(), LinearRegression()).fit(X, y)
lin_invariant = np.allclose(lin_raw.predict(X), lin_scaled.predict(X))
print("identical predictions:", lin_invariant)
print("raw coefficients   :", lin_raw.coef_.round(4), " <- very different for different units")
print("scaled coefficients:", lin_scaled[-1].coef_.round(3))

ols_contributions = pd.Series(np.abs(lin_raw.coef_) * stds.to_numpy(), index=X.columns)
ols_contributions.round(2)

In [ ]:
ridge_raw = Ridge(alpha=200).fit(X, y)
ridge_scaled = make_pipeline(StandardScaler(), Ridge(alpha=200)).fit(X, y)
ridge_raw_contributions = pd.Series(np.abs(ridge_raw.coef_) * stds.to_numpy(), index=X.columns)
ridge_scaled_contributions = pd.Series(np.abs(ridge_scaled[-1].coef_), index=X.columns)
raw_ratio = ridge_raw_contributions["x1"] / ridge_raw_contributions["x3"]
scaled_ratio = ridge_scaled_contributions["x1"] / ridge_scaled_contributions["x3"]

print(pd.DataFrame({"OLS": ols_contributions, "Ridge raw": ridge_raw_contributions,
                    "Ridge scaled": ridge_scaled_contributions}).round(2))
print(f"x1/x3 ratio: raw {raw_ratio:.2f} | scaled {scaled_ratio:.2f}")

With alpha = 200 the contribution of x1 is halved and that of x3 stays intact, when they should have shrunk
the same way: the penalty is on `β²`, and for x3 (measured in large units) a tiny `β` is enough to have the same
effect, so the penalty does not "feel" it. With the scaler all three coefficients are in comparable units.

In [ ]:
alphas = np.logspace(-1, 10, 111)
half_alpha = pd.DataFrame(index=["raw", "scaled"], columns=["x1", "x2", "x3"], dtype=float)
for a in alphas:
    c_raw = np.abs(Ridge(alpha=a).fit(X, y).coef_) * stds.to_numpy()
    c_scaled = np.abs(make_pipeline(StandardScaler(), Ridge(alpha=a)).fit(X, y)[-1].coef_)
    for row, c in [("raw", c_raw), ("scaled", c_scaled)]:
        for j, col in enumerate(X.columns):
            if np.isnan(half_alpha.loc[row, col]) and c[j] < ols_contributions[col] / 2:
                half_alpha.loc[row, col] = a
pd.options.display.float_format = "{:.3g}".format
half_alpha

In [ ]:
answer = "b"

## Check

In [ ]:
with verify():
    check(bool(lin_invariant), "OLS is scale invariant: same predictions with and without the scaler",
          "the predictions of LinearRegression should coincide with and without StandardScaler")
    check(np.allclose(ols_contributions, 2.0, atol=0.5),
          "in comparable units the three features have the same contribution (about 2)", "contribution = |coef| * std of each feature")
    check(raw_ratio < 0.65, "without scaling Ridge shrinks x1 much more than x3",
          "the contribution of x1 should be reduced to about half of that of x3")
    check(0.85 < scaled_ratio < 1.2, "with scaling the three features are shrunk equally",
          "after scaling the ratio between the contributions should be about 1")
    check(half_alpha.loc["raw", "x3"] / half_alpha.loc["raw", "x1"] > 1e5,
          "without scaling the alpha that halves x3 is over a hundred thousand times the one that halves x1",
          "without scaling the scale of alpha needed should change by many orders of magnitude from x1 to x3")
    check(half_alpha.loc["scaled"].max() / half_alpha.loc["scaled"].min() < 3,
          "with scaling alpha has the same effect on all the features", "with scaling the three alphas should be about equal")
    check(answer == "b", "correct explanation", "reread the half_alpha table")
done()

## What you learned

- OLS is scale invariant: if you multiply a feature by 1000, its coefficient is divided by 1000 and the
  prediction does not change. Ridge is not: the penalty α·β² depends on the numeric value of the coefficient, so on the unit.
- Without scaling you need alpha values **almost 6 orders of magnitude** apart (from about 250 to about 1.6·10⁸) to
  halve x1, x2 and x3, even though they have the same importance. A single alpha, chosen in cross-validation, cannot fit them all: it chooses a compromise
  that distorts the selection, in a way that depends on how you measured the features.
- With the scaler the three alphas coincide (about the number of observations, 200): it is the theoretical reason why the "right"
  penalty is defined on the standardized features.

## Common pitfall

Scaling the features **before** the split or the cross-validation instead of inside the Pipeline (leakage: E3 and the
lessons), or scaling the target without noticing. And the opposite: forgetting the scaler because "the features are
all numeric already". The unit of measure always matters, for any model with a penalty on the weights.

## Interview question

> *Click on a question to show its answer.*

<details><summary>Why does linear regression not need the scaler but Ridge does?</summary>

OLS minimizes the squared errors without constraining the coefficients: rescaling a feature rescales its coefficient
in the opposite direction and the optimum stays the same, so the predictions do not change. Ridge adds α·Σβ²: it is a penalty
on the *numeric value* of the coefficients, which depends on the unit of measure. At equal real effect, a feature in
small units has a large coefficient and is punished more. The scaler makes the coefficients comparable.

</details>